In [2]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

# 1. Orchestration Environment Setup
def setup_distributed_process(rank, world_size):
    """
    rank: Unique identifier for the current GPU process (0, 1, 2...)
    world_size: Total number of GPUs participating in the run
    """
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    # Initialize the default backend (NCCL is the optimal framework for NVIDIA hardware)
    dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    dist.destroy_process_group()

# 2. Execution Run Engine
def train_ddp_worker(rank, world_size):
    setup_distributed_process(rank, world_size)
    
    # Instantiate raw model architecture
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 2)
    ).to(rank) # Push directly to the specific isolated GPU
    
    # Wrap the model in the DDP structural wrapper
    # This automatically syncs weights initially and establishes the All-Reduce hooks
    ddp_model = DDP(model, device_ids=[rank])
    
    # Construct synthetic data matrix
    features = torch.randn(500, 20)
    labels = torch.randint(0, 2, (500,))
    dataset = torch.utils.data.TensorDataset(features, labels)
    
    # Distributed Data Sampler prevents different GPUs from reading duplicate samples
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=32, sampler=sampler)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(ddp_model.parameters(), lr=0.001)
    
    # Execution Training Iteration Loop
    ddp_model.train()
    for epoch in range(1):
        # CRITICAL STEP: Set the epoch inside the sampler to ensure shuffling varies across epochs
        sampler.set_epoch(epoch)
        
        for batch_X, batch_y in loader:
            # Route inputs to local processor VRAM
            batch_X = batch_X.to(rank)
            batch_y = batch_y.to(rank)
            
            optimizer.zero_grad()
            outputs = ddp_model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward() # Automatically triggers multi-GPU gradient synchronization via All-Reduce
            optimizer.step()
            
    if rank == 0:
        print("Training successfully executed on Master process node.")
        
    cleanup()

# Production deployment invocation pattern via the shell terminal:
# torchrun --nproc_per_node=2 distributed_script.py